---
title: "Chapter 8: Loss Functions, Multiclass Classification, and Conclusions"
---

::: {.callout-lo}
By the end of this chapter, you should be able to:

- Compare how squared error and absolute error penalize regression mistakes.
- Distinguish a loss function, a regularized fitting objective, and an evaluation score.
- Explain why accuracy and log loss can rank probability predictions differently.
- Explain how multinomial logistic regression turns class scores into probabilities.
- Fit and evaluate a multiclass logistic-regression pipeline.
- Critique a supervised learning workflow from problem formulation to final evaluation.
:::

Chapter 7 introduced linear models and the losses used to fit them. We now look more closely at what a loss rewards. Two models can make the same class predictions yet assign very different probabilities, and two regression models can have similar typical errors while differing in their largest mistakes. How we measure those differences affects which model we prefer.

We will then extend logistic regression from two classes to several classes. The chapter ends by returning to the central task of this course: building a workflow that gives us credible evidence about predictions on new data.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## Which mistakes should count most?

Suppose two models predict delivery times in minutes. On four validation examples, model A's prediction errors are 0, 0, 0, and 8 minutes, while model B's errors are 3, 3, 3, and 3 minutes. Model A is usually exact but makes one larger mistake. Model B is consistently a little wrong. Which would you prefer?

The answer depends partly on the consequences of an error. **Mean absolute error (MAE)** averages the magnitudes of the errors. **Mean squared error (MSE)** averages their squares. Let $e_i=\hat y_i-y_i$ denote prediction minus observed target for example $i$. With $n$ examples,

$$
\operatorname{MAE}=\frac{1}{n}\sum_{i=1}^n |e_i|,
\qquad
\operatorname{MSE}=\frac{1}{n}\sum_{i=1}^n e_i^2.
$$

The summation symbol means to add the contribution from each example. Both measures are zero for perfect predictions and increase as errors grow. Neither distinguishes an early prediction from a late one of the same magnitude.

In [2]:
errors = pd.DataFrame({"Model A": [0, 0, 0, 8], "Model B": [3, 3, 3, 3]})
pd.DataFrame({
    "MAE (minutes)": errors.abs().mean(),
    "MSE (minutes squared)": errors.pow(2).mean(),
})

,MAE (minutes),MSE (minutes squared)
Model A,2.0,16.0
Model B,3.0,9.0


MAE favours model A: its average absolute error is 2 minutes, compared with 3 for model B. MSE favours model B: its mean squared error is 9, compared with 16 for model A. Squaring makes the single 8-minute error contribute 64 to the sum.

Doubling an error doubles its absolute loss but quadruples its squared loss. Squared error therefore gives large mistakes relatively more influence. This can be useful when large errors are especially costly, but it also makes the fitting procedure more sensitive to unusual target values. Absolute error grows more slowly; it does not make unusual observations irrelevant.

MAE is expressed in the target's units, while MSE is expressed in squared units. Do not compare their numerical values as though a smaller number across different metrics identifies the better metric. Choose a measure whose behaviour helps answer the application question. If predicting a delivery too early has a different cost from predicting it too late, neither of these symmetric losses fully captures that difference.

## Loss, fitting objective, and evaluation score

A **loss** measures disagreement between a prediction and its target. A **fitting objective** combines the losses over training examples and may add a regularization penalty. An **evaluation score** summarizes performance on the data we use to assess the fitted model. A loss can also serve as an evaluation measure: for example, we can compute MSE on validation predictions.

For scikit-learn's `Ridge`, the fitting objective is

$$
\sum_{i=1}^n (\hat y_i-y_i)^2 + \alpha\sum_{j=1}^d w_j^2.
$$

Here, $d$ is the number of features, $w_j$ is a coefficient, and $\alpha$ controls the penalty. The first term is a **sum** of squared errors, rather than a mean. Without a penalty, minimizing the sum or the mean gives the same fitted predictions. With a penalty, changing the scale of the loss changes its balance with the penalty unless we also rescale $\alpha$. This is one reason to check an estimator's exact objective before interpreting a regularization value. See the [Ridge documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html).

Changing the `scoring` argument of a search does not replace the estimator's fitting objective. For example, comparing `Ridge` models using validation MAE still fits each candidate using its squared-error objective and penalty. The search chooses among those fitted candidates according to MAE.

Scikit-learn's search tools maximize scores. For losses, scorer names such as `neg_mean_absolute_error` and `neg_log_loss` return the negative of the loss so that larger is better. A score of $-2$ is better than $-5$ because it corresponds to a smaller loss. Estimator defaults also differ: `Ridge.score` reports $R^2$, while `LogisticRegression.score` reports accuracy. Always establish which quantity you are comparing.

### Exercise 8.1: What changed inside the search?

A colleague changes `GridSearchCV(ridge_pipeline, ..., scoring="r2")` to use `scoring="neg_mean_absolute_error"`. They conclude that `Ridge.fit` now minimizes absolute error. Explain the mistake. Could the selected `alpha` still change?

::: {.callout-tip collapse="true" title="Solution"}
Each candidate still fits using Ridge's squared-error objective and regularization penalty. The search now compares its validation predictions using negative MAE, so the ranking of candidates—and the selected `alpha`—can change. We should choose the comparison measure before inspecting final test results.
:::

## Evaluating probabilities with log loss

Accuracy uses the predicted class, so it cannot distinguish two models that make identical class predictions. Log loss uses the probability assigned to the observed class. As in Chapter 7, the loss for one example is

$$
-\log(p_{\text{observed}}),
$$

where $\log$ is the natural logarithm. Giving the observed class a probability near 1 produces a loss near zero. Assigning it a probability close to zero produces a large loss.

Consider four messages whose observed labels are not spam, not spam, spam, and spam. Before running the code, compare the two sets of spam probabilities. Both models predict all four labels correctly using a 0.5 threshold. Which assigns more probability to the observed outcomes?

In [3]:
y_messages = np.array([0, 0, 1, 1])
spam_probabilities = {
    "Hesitant": np.array([0.4, 0.4, 0.6, 0.6]),
    "Confident": np.array([0.1, 0.1, 0.9, 0.9]),
}
rows = []
for name, p_spam in spam_probabilities.items():
    probabilities = np.column_stack([1 - p_spam, p_spam])
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_messages, p_spam > 0.5),
        "Log loss": log_loss(y_messages, probabilities, labels=[0, 1]),
    })
pd.DataFrame(rows).set_index("Model")

,Accuracy,Log loss
Model,,
Hesitant,1.0,0.510826
Confident,1.0,0.105361


Both models have accuracy 1, but the confident model has lower log loss. This does not mean that making probabilities more extreme always helps. Confidence helps here because these predictions are correct. If the last message were actually not spam, assigning it a spam probability of 0.99 would incur much more loss than assigning it 0.6.

Log loss evaluates more than the hard class decision, but a low value on one dataset does not guarantee reliable probabilities for every group or for future data. The same concerns about representative evaluation data and overfitting still apply.

## More than two classes

Suppose we want to route a support request to billing, technical support, or account management. Each request has one of three possible labels. This is **multiclass classification**. It differs from a problem in which one request can receive several labels at once; here, exactly one class is the target for each example.

**Multinomial logistic regression** extends logistic regression by computing a linear score for each class. For class $k$,

$$
z_k=\mathbf{w}_k^{\mathsf T}\mathbf{x}+b_k.
$$

Each class has its own coefficients and intercept. We convert the scores into a single set of probabilities using **softmax**:

$$
p_k=\frac{e^{z_k}}{\sum_{j=1}^K e^{z_j}},
$$

where $K$ is the number of classes. Exponentiating makes every contribution positive, and dividing by their sum makes the probabilities sum to 1. A class with a higher score receives a higher probability. It is the relative scores that matter: increasing one class's score while holding the others fixed raises its probability and lowers the others.

For example, scores of 2, 1, and 0 give the following probabilities.

In [4]:
scores = np.array([2.0, 1.0, 0.0])
# Subtracting the maximum leaves softmax unchanged and avoids large exponentials.
weights = np.exp(scores - scores.max())
pd.Series(weights / weights.sum(), index=["billing", "technical", "account"])

billing      0.665241
technical    0.244728
account      0.090031
dtype: float64

The largest probability is approximately 0.665, so the predicted class is billing. A multiclass prediction does not require any class to exceed 0.5: probabilities of 0.40, 0.35, and 0.25 would also select billing.

Applying a separate sigmoid to every score would not generally produce probabilities that sum to 1. Softmax makes the classes compete within one distribution. With two classes, softmax's probability for the second class is the sigmoid of the difference between its score and the first class's score, connecting this construction to binary logistic regression.

Fitting uses the same observed-class log loss as before, now with more possible classes, together with regularization. For an observed technical-support request, probabilities of 0.1, 0.8, and 0.1 yield a loss of $-\log(0.8)\approx0.223$. The label's numeric encoding does not represent an order or a distance between classes.

### A complete multiclass example

We will use scikit-learn's small Iris dataset, which contains four flower measurements and a target identifying one of three iris species. It is included with scikit-learn and requires no download. It is useful for tracing the workflow, though success on this small dataset is limited evidence about other classification tasks.

We reserve test data before tuning. `stratify=y` keeps class proportions approximately the same in the two subsets. We then search over `C` with preprocessing inside the pipeline. The `lbfgs` solver fits multinomial logistic regression for this three-class target; see the [LogisticRegression documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html).

In [5]:
iris = load_iris(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.25, random_state=571, stratify=iris.target
)
pipeline = make_pipeline(
    StandardScaler(), LogisticRegression(solver="lbfgs", max_iter=1000)
)
search = GridSearchCV(
    pipeline,
    {"logisticregression__C": [0.1, 1.0, 10.0]},
    scoring="neg_log_loss",
    cv=5,
)
search.fit(X_train, y_train)
print("Selected settings:", search.best_params_)
print("Selected mean validation log loss:", -search.best_score_)

Selected settings: {'logisticregression__C': 10.0}
Selected mean validation log loss: 0.08148409214713796


The search refits the selected pipeline on all training data. We now evaluate that model and a baseline that always returns the training class proportions. We selected log loss as our primary comparison measure because this example focuses on probabilities; we also report accuracy to describe the hard predictions. We do not use either test result to restart tuning.

In [6]:
baseline = DummyClassifier(strategy="prior").fit(X_train, y_train)
results = []
for name, model in [("Baseline", baseline), ("Logistic regression", search.best_estimator_)]:
    results.append({
        "Model": name,
        "Test accuracy": accuracy_score(y_test, model.predict(X_test)),
        "Test log loss": log_loss(y_test, model.predict_proba(X_test), labels=model.classes_),
    })
pd.DataFrame(results).set_index("Model")

,Test accuracy,Test log loss
Model,,
Baseline,0.315789,1.099159
Logistic regression,0.947368,0.111436


In [7]:
model = search.best_estimator_
probabilities = model.predict_proba(X_test.iloc[:5])
class_names = iris.target_names[model.classes_]
prediction_table = pd.DataFrame(probabilities, columns=class_names)
prediction_table["Predicted"] = iris.target_names[model.predict(X_test.iloc[:5])]
prediction_table["Observed"] = iris.target_names[y_test.iloc[:5].to_numpy()]
prediction_table

,setosa,versicolor,virginica,Predicted,Observed
0,1.577641e-06,0.014345,9.856533e-01,virginica,virginica
1,9.953823e-01,0.004618,4.904486e-13,setosa,setosa
2,9.980563e-01,0.001944,2.130821e-13,setosa,setosa
3,8.853663e-02,0.911299,1.639312e-04,versicolor,versicolor
4,1.383787e-08,0.001374,9.986261e-01,virginica,virginica


Each probability column corresponds to an entry of `classes_`; we use that order when attaching species names. The largest probability identifies the predicted species. The model still has linear pairwise decision boundaries in the represented feature space, so adding classes does not remove the limitations of a linear model.

### Exercise 8.2: Same class prediction, different evidence

Two models assign probabilities to billing, technical support, and account management. Model A returns (0.40, 0.35, 0.25), while model B returns (0.90, 0.05, 0.05).

What does each model predict? Which receives lower log loss if the observed class is billing? What if it is technical support? Explain why accuracy alone cannot distinguish these models on this request.

::: {.callout-tip collapse="true" title="Solution"}
Both predict billing. If billing is observed, model B has lower loss because it assigned 0.90 to that class rather than 0.40. If technical support is observed, model A has lower loss because it assigned 0.35 rather than 0.05. Accuracy counts both as correct in the first case and both as incorrect in the second; it ignores their probabilities.
:::

## Bringing the workflow together

The models in this book use different prediction rules, but the questions surrounding them recur. What is one example? What are we predicting? Which features will be available at prediction time? What simple baseline should the model improve on? An unclear answer to one of these questions can undermine a technically correct call to `fit`.

Once the task is defined, reserve data for final evaluation and keep learned preprocessing inside the workflow used for cross-validation. Compare plausible models and hyperparameters using training data, with a measure that reflects what you need predictions to do. A larger search is not automatically better evidence: repeated comparisons can make the winning validation result optimistic.

After choosing the workflow, evaluate it on the held-out test data and explain the result relative to a baseline. Report limitations as part of the result: which mistakes remain, how the evaluation data might differ from future cases, and what the analysis cannot establish. A predictive association does not establish causation, and a strong score does not by itself justify using a model to make consequential decisions.

### Exercise 8.3: Review a proposed workflow

A team wants to route support requests to three departments. It includes a field recording which department eventually resolved each request, builds a text vocabulary using all requests, and then creates a train/test split. It tries many values of `C`, keeps the one with the best test accuracy, and reports that accuracy as evidence that its probability predictions are reliable.

Identify the problems and propose a corrected workflow using tools from this book. Explain which choices must be settled before final test evaluation.

::: {.callout-tip collapse="true" title="Solution"}
The resolving department is not available when a new request arrives and would reveal information about the outcome. Remove it and check the availability of the remaining inputs. Reserve the test set before learning a vocabulary; place `CountVectorizer` and the model in a pipeline so that each cross-validation fold learns its vocabulary only from that fold's training portion. Use training-data cross-validation to choose `C`, and compare against a baseline. Decide whether the intended use requires hard routing decisions, probability estimates, or both, and choose evaluation measures accordingly. Accuracy alone does not assess the assigned probabilities. Evaluate the selected workflow once on the test set and explain the limits of that evidence.
:::

You now have the components of a supervised learning workflow: a clearly defined target, a suitable representation, a baseline, candidate models, and a disciplined way to compare them. Using these components well means checking the assumptions around the code as carefully as the code itself. The next time you face a new prediction problem, start with those questions and build the simplest workflow that can provide useful evidence.